In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# QUANTITATIVE BEHAVIORAL FINANCE FRAMEWORK — LIVE GOLD/GLD DATA
# ═══════════════════════════════════════════════════════════════
# Complete Jupyter Notebook - Run All Cells

# %% [markdown]
# # Quantitative Behavioral Finance — Framework Portfolio
# ## *Models & Frameworks with Live GLD Data*

# %% Cell 1: Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.gridspec as gridspec
from scipy.stats import norm, kurtosis, skew
from scipy.optimize import minimize
from scipy.linalg import eigh
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# Install yfinance if needed (uncomment if not installed)
# !pip install yfinance

import yfinance as yf

# ── GLOBAL DARK THEME ──────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0a0a0f',
    'axes.facecolor': '#0d0d1a',
    'axes.edgecolor': '#2a2a4a',
    'axes.labelcolor': '#c8c8ff',
    'axes.titlecolor': '#ffffff',
    'axes.grid': True,
    'grid.color': '#1a1a3a',
    'grid.linewidth': 0.5,
    'xtick.color': '#8888cc',
    'ytick.color': '#8888cc',
    'text.color': '#c8c8ff',
    'figure.titlesize': 16,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'monospace',
    'lines.linewidth': 1.8,
    'legend.facecolor': '#111122',
    'legend.edgecolor': '#3a3a6a',
    'legend.labelcolor': '#c8c8ff',
})

# ── QUANTUM COLOR PALETTES ─────────────────────────────────────
QUANTUM_BLUE = '#00d4ff'
QUANTUM_PURPLE = '#b444ff'
QUANTUM_GREEN = '#00ff88'
QUANTUM_GOLD = '#ffd700'
QUANTUM_RED = '#ff4455'
QUANTUM_PINK = '#ff44aa'

NEON_CMAP = mcolors.LinearSegmentedColormap.from_list('neon',
    ['#0a0a2e', '#1a0050', '#4400cc', '#0088ff', '#00ffcc', '#ffffff'])
FIRE_CMAP = mcolors.LinearSegmentedColormap.from_list('fire',
    ['#0a0010', '#3a0030', '#cc0044', '#ff4400', '#ffaa00', '#ffff44'])
PLASMA_CMAP = mcolors.LinearSegmentedColormap.from_list('plasma2',
    ['#0a0a1a', '#220066', '#8800aa', '#ff00cc', '#ff88ff', '#ffffff'])

np.random.seed(42)
print('✦ Environment loaded — Dark Quantum Theme Active ✦')
print(f'✦ NumPy {np.__version__} | Pandas {pd.__version__} ✦')

# %% Cell 2: Fetch Live GLD Data
print('⟳ Fetching live GLD data from Yahoo Finance...')

# Fetch GLD ETF data (2 years of daily data)
gld = yf.Ticker("GLD")
gld_hist = gld.history(period="2y")

# Also fetch gold futures for comparison
gc = yf.Ticker("GC=F")
gc_hist = gc.history(period="2y")

# Clean and prepare data
gld_prices = gld_hist['Close'].dropna()
gld_returns = gld_prices.pct_change().dropna()
gld_log_returns = np.log(gld_prices / gld_prices.shift(1)).dropna()

print(f'✓ GLD data fetched: {len(gld_prices)} trading days')
print(f'✓ Date range: {gld_prices.index[0].strftime("%Y-%m-%d")} to {gld_prices.index[-1].strftime("%Y-%m-%d")}')
print(f'✓ Current GLD price: ${gld_prices.iloc[-1]:.2f}')
print(f'✓ GLD Annualized Return: {gld_returns.mean() * 252 * 100:.2f}%')
print(f'✓ GLD Annualized Volatility: {gld_returns.std() * np.sqrt(252) * 100:.2f}%')

# Store key statistics for later modules
GLD_MU = gld_returns.mean()
GLD_SIGMA = gld_returns.std()
GLD_PRICES = gld_prices.values
GLD_RETURNS = gld_returns.values
GLD_LOG_RETURNS = gld_log_returns.values

# %% [markdown]
# ---
# ## MODULE 01 — Heston Stochastic Volatility Model
# ### Implied Volatility Surface · 3D Quantum Render (Calibrated to GLD)

# %% Cell 3: Module 01 - Heston Model
import math

def heston_cf(phi, S0, K, T, r, kappa, theta, xi, rho, v0):
    """Characteristic function for Heston model"""
    x = np.log(S0 / K)
    a = kappa * theta
    u1, u2 = 0.5, -0.5
    b1, b2 = kappa - rho * xi, kappa
    
    d1 = np.sqrt((rho*xi*1j*phi - b1)**2 - xi**2*(2*u1*1j*phi - phi**2) + 1e-10)
    d2 = np.sqrt((rho*xi*1j*phi - b2)**2 - xi**2*(2*u2*1j*phi - phi**2) + 1e-10)
    
    g1 = (b1 - rho*xi*1j*phi + d1) / (b1 - rho*xi*1j*phi - d1 + 1e-10)
    g2 = (b2 - rho*xi*1j*phi + d2) / (b2 - rho*xi*1j*phi - d2 + 1e-10)
    
    exp_d1T = np.exp(d1*T)
    exp_d2T = np.exp(d2*T)
    
    C1 = r*1j*phi*T + (a/xi**2)*((b1-rho*xi*1j*phi+d1)*T - 2*np.log((1-g1*exp_d1T)/(1-g1) + 1e-10))
    C2 = r*1j*phi*T + (a/xi**2)*((b2-rho*xi*1j*phi+d2)*T - 2*np.log((1-g2*exp_d2T)/(1-g2) + 1e-10))
    
    D1 = ((b1-rho*xi*1j*phi+d1)/xi**2)*((1-exp_d1T)/(1-g1*exp_d1T + 1e-10))
    D2 = ((b2-rho*xi*1j*phi+d2)/xi**2)*((1-exp_d2T)/(1-g2*exp_d2T + 1e-10))
    
    f1 = np.exp(C1 + D1*v0 + 1j*phi*x)
    f2 = np.exp(C2 + D2*v0 + 1j*phi*x)
    return f1, f2

def heston_price(S0, K, T, r, kappa, theta, xi, rho, v0, option_type='call'):
    """Price via numerical integration of characteristic function"""
    n, dphi = 2000, 0.05
    phi_arr = np.linspace(1e-6, n*dphi, n)
    P1, P2 = 0.5, 0.5
    for phi in phi_arr:
        f1, f2 = heston_cf(phi, S0, K, T, r, kappa, theta, xi, rho, v0)
        P1 += (1/np.pi) * np.real(f1 / (1j*phi)) * dphi
        P2 += (1/np.pi) * np.real(f2 / (1j*phi)) * dphi
    price = S0*P1 - K*np.exp(-r*T)*P2
    return max(price, 0)

def bs_iv(price, S, K, T, r, flag='c'):
    """Bisection method for implied vol extraction"""
    lo, hi = 1e-6, 5.0
    for _ in range(100):
        mid = (lo + hi) / 2
        d1 = (np.log(S/K) + (r + 0.5*mid**2)*T) / (mid*np.sqrt(T) + 1e-10)
        d2 = d1 - mid*np.sqrt(T)
        p = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
        if p < price:
            lo = mid
        else:
            hi = mid
        if abs(hi - lo) < 1e-7:
            break
    return mid

# ── Parameters calibrated to GLD ─────────────────────────────────
S0 = GLD_PRICES[-1]  # Current GLD price
r = 0.05  # Risk-free rate
# Heston parameters calibrated to gold volatility characteristics
realized_vol = GLD_SIGMA * np.sqrt(252)
kappa = 2.5  # Mean reversion speed
theta = realized_vol**2  # Long-run variance (from GLD data)
xi = 0.5  # Vol of vol
rho = -0.3  # Gold typically has less negative correlation than equities
v0 = theta  # Current variance

strikes = np.linspace(S0 * 0.7, S0 * 1.3, 20)
maturities = np.linspace(0.1, 2.0, 18)
K_grid, T_grid = np.meshgrid(strikes, maturities)
moneyness = np.log(K_grid / S0)

print(f'⟳ Computing Heston implied vol surface for GLD (S0=${S0:.2f})...')
IV_surface = np.zeros_like(K_grid)

for i, T in enumerate(maturities):
    for j, K in enumerate(strikes):
        try:
            p = heston_price(S0, K, T, r, kappa, theta, xi, rho, v0)
            IV_surface[i, j] = bs_iv(p, S0, K, T, r) * 100
        except:
            IV_surface[i, j] = np.sqrt(theta) * 100

# ── 3D QUANTUM SURFACE PLOT ─────────────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor='#0a0a0f')

# Main 3D surface
ax1 = fig.add_subplot(121, projection='3d', facecolor='#0a0a0f')
surf = ax1.plot_surface(moneyness, T_grid, IV_surface, cmap=NEON_CMAP,
                        alpha=0.88, linewidth=0, antialiased=True)
ax1.contourf(moneyness, T_grid, IV_surface, zdir='z', offset=IV_surface.min(),
             cmap=NEON_CMAP, alpha=0.3, levels=15)

ax1.set_xlabel('Log-Moneyness log(K/S)', labelpad=8)
ax1.set_ylabel('Maturity T (years)', labelpad=8)
ax1.set_zlabel('Implied Vol (%)', labelpad=8)
ax1.set_title(f'GLD HESTON IMPLIED VOLATILITY SURFACE (S₀=${S0:.2f})', pad=15, fontsize=12,
              color=QUANTUM_GOLD, fontweight='bold')
ax1.tick_params(colors='#6666aa', labelsize=8)
ax1.xaxis.pane.fill = False
ax1.yaxis.pane.fill = False
ax1.zaxis.pane.fill = False
ax1.xaxis.pane.set_edgecolor('#1a1a3a')
ax1.yaxis.pane.set_edgecolor('#1a1a3a')
ax1.zaxis.pane.set_edgecolor('#1a1a3a')

cb = fig.colorbar(surf, ax=ax1, shrink=0.4, pad=0.1)
cb.ax.yaxis.set_tick_params(color='#8888cc')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='#8888cc', fontsize=8)
cb.set_label('IV (%)', color='#8888cc', fontsize=9)

# Smile cross-sections
ax2 = fig.add_subplot(122, facecolor='#0d0d1a')
smile_colors = plt.cm.cool(np.linspace(0.1, 0.9, len(maturities[::3])))
for idx, (i, col) in enumerate(zip(range(0, len(maturities), 3), smile_colors)):
    ax2.plot(moneyness[i], IV_surface[i], color=col, lw=2,
             label=f'T = {maturities[i]:.1f}y', alpha=0.9)
ax2.axvline(0, color=QUANTUM_GOLD, lw=1.2, ls='--', alpha=0.6, label='ATM')
ax2.set_xlabel('Log-Moneyness log(K/S)')
ax2.set_ylabel('Implied Volatility (%)')
ax2.set_title('GLD VOLATILITY SMILE CROSS-SECTIONS', color=QUANTUM_PURPLE)
ax2.legend(fontsize=8, ncol=2)
ax2.fill_between(moneyness[0], IV_surface.min(axis=0), IV_surface.max(axis=0),
                 alpha=0.08, color=QUANTUM_GOLD)

plt.suptitle(f'MODULE 01 · HESTON STOCHASTIC VOLATILITY · κ={kappa:.1f} θ={theta:.4f} ξ={xi:.1f} ρ={rho:.1f}',
             color='#ffffff', fontsize=10, y=0.02)
plt.tight_layout()
plt.show()
print('✓ Heston surface rendered with live GLD data')

# %% [markdown]
# ---
# ## MODULE 02 — Behavioral Alpha: Momentum × Mean Reversion Regime
# ### Cross-Sectional Momentum with Regime-Switching Detection (Live GLD)

# %% Cell 4: Module 02 - Momentum/Mean Reversion
def compute_momentum_signal(returns, lookback=63, skip=5):
    """Risk-adjusted momentum (Sharpe-ratio based signal)"""
    signal = np.zeros(len(returns))
    for t in range(lookback + skip, len(returns)):
        window = returns[t-lookback-skip : t-skip]
        signal[t] = window.mean() / (window.std() + 1e-8)
    return signal

def compute_mr_signal(returns, lookback=20):
    """Mean-reversion z-score signal"""
    signal = np.zeros(len(returns))
    for t in range(lookback, len(returns)):
        w = returns[t-lookback:t]
        signal[t] = -(returns[t-1] - w.mean()) / (w.std() + 1e-8)
    return signal

# Use live GLD data
rets = GLD_RETURNS
prices = GLD_PRICES

# Compute signals
mom_sig = compute_momentum_signal(rets)
mr_sig = compute_mr_signal(rets)

# Regime detection based on realized volatility
realized_vol_rolling = pd.Series(rets).rolling(21).std().values
vol_threshold = np.nanpercentile(realized_vol_rolling, 70)
regime_weight = np.where(realized_vol_rolling > vol_threshold, 0.2, 0.8)

# Detect regimes: 0 = trending (low vol), 1 = mean-reverting/stressed (high vol)
regimes = (realized_vol_rolling > vol_threshold).astype(int)

# Combined signal
combined = regime_weight * mom_sig + (1 - regime_weight) * mr_sig

# Strategy returns
pos = np.sign(combined)
strat_rets = pos[:-1] * rets[1:] * 0.5
cum_strat = np.cumprod(1 + strat_rets)
cum_buy = np.cumprod(1 + rets[1:])

# ── PLOT ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35,
                       left=0.07, right=0.96, top=0.92, bottom=0.06)

t_ax = np.arange(len(rets))

# 1. Price + regime overlay
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(prices, color=QUANTUM_GOLD, lw=1.2, label='GLD Price')
ax1.fill_between(t_ax, prices.min(), prices.max(),
                 where=(regimes==1), alpha=0.18, color=QUANTUM_RED, label='High Vol Regime')
ax1.fill_between(t_ax, prices.min(), prices.max(),
                 where=(regimes==0), alpha=0.07, color=QUANTUM_GREEN, label='Low Vol Regime')
ax1.set_title('LIVE GLD PRICE PATH WITH VOLATILITY REGIME DETECTION', color=QUANTUM_GOLD)
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left', fontsize=8)

# 2. Signals
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(mom_sig, color=QUANTUM_GREEN, lw=1, alpha=0.8, label='Momentum')
ax2.plot(mr_sig, color=QUANTUM_PINK, lw=1, alpha=0.8, label='Mean Rev')
ax2.plot(combined, color=QUANTUM_GOLD, lw=1.5, label='Regime-Combo')
ax2.axhline(0, color='#444466', lw=0.8, ls='--')
ax2.set_title('SIGNAL DECOMPOSITION', color=QUANTUM_PURPLE)
ax2.legend(fontsize=8)
ax2.set_ylim(-5, 5)

# 3. Cumulative PnL
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(cum_strat, color=QUANTUM_GREEN, lw=2.0, label='Regime-Aware Strategy')
ax3.plot(cum_buy, color='#4444aa', lw=1.2, ls='--', label='Buy & Hold GLD')
ax3.fill_between(range(len(cum_strat)), 1, cum_strat,
                 where=(cum_strat >= 1), alpha=0.15, color=QUANTUM_GREEN)
ax3.fill_between(range(len(cum_strat)), 1, cum_strat,
                 where=(cum_strat < 1), alpha=0.2, color=QUANTUM_RED)
ax3.axhline(1, color='#555577', lw=0.8, ls='--')
ax3.set_title('CUMULATIVE PnL vs BUY & HOLD GLD', color=QUANTUM_GREEN)
ax3.legend(fontsize=8)

# 4. Return distribution by regime
ax4 = fig.add_subplot(gs[2, 0])
r_low_vol = rets[regimes == 0]
r_high_vol = rets[regimes == 1]
bins = np.linspace(-0.06, 0.06, 60)
if len(r_low_vol) > 0:
    ax4.hist(r_low_vol, bins=bins, color=QUANTUM_GREEN, alpha=0.6, density=True,
             label=f'Low Vol (σ={r_low_vol.std()*100:.2f}%)')
if len(r_high_vol) > 0:
    ax4.hist(r_high_vol, bins=bins, color=QUANTUM_RED, alpha=0.6, density=True,
             label=f'High Vol (σ={r_high_vol.std()*100:.2f}%)')
ax4.set_title('GLD RETURN DISTRIBUTIONS BY REGIME', color=QUANTUM_GOLD)
ax4.set_xlabel('Daily Return')
ax4.legend(fontsize=8)

# 5. Rolling Sharpe
ax5 = fig.add_subplot(gs[2, 1])
roll_sharpe = pd.Series(strat_rets).rolling(63).apply(
    lambda x: x.mean() / (x.std() + 1e-8) * np.sqrt(252))
ax5.plot(roll_sharpe, color=QUANTUM_PURPLE, lw=1.5)
ax5.axhline(0, color='#555577', lw=0.8, ls='--')
ax5.axhline(1, color=QUANTUM_GOLD, lw=0.8, ls=':', alpha=0.7, label='Sharpe = 1')
ax5.fill_between(range(len(roll_sharpe)), 0, roll_sharpe,
                 where=(roll_sharpe >= 0), alpha=0.2, color=QUANTUM_PURPLE)
ax5.fill_between(range(len(roll_sharpe)), 0, roll_sharpe,
                 where=(roll_sharpe < 0), alpha=0.2, color=QUANTUM_RED)
ax5.set_title('ROLLING 63-DAY SHARPE RATIO', color=QUANTUM_PURPLE)
ax5.legend(fontsize=8)

ann_sharpe = strat_rets.mean() / (strat_rets.std() + 1e-8) * np.sqrt(252)
max_dd = (pd.Series(cum_strat) / pd.Series(cum_strat).cummax() - 1).min()
fig.suptitle(f'MODULE 02 · GLD BEHAVIORAL ALPHA · Ann.Sharpe={ann_sharpe:.2f} MaxDD={max_dd*100:.1f}%',
             color='#ffffff', fontsize=10, y=0.98)
plt.show()
print(f'✓ GLD Strategy Sharpe: {ann_sharpe:.3f} | Max Drawdown: {max_dd*100:.2f}%')

# %% [markdown]
# ---
# ## MODULE 03 — Prospect Theory & Behavioral Utility
# ### Kahneman-Tversky Value Function + Probability Weighting (GLD Context)

# %% Cell 5: Module 03 - Prospect Theory
def kt_value(x, alpha=0.88, beta=0.88, lam=2.25):
    """Kahneman-Tversky value function"""
    return np.where(x >= 0, x**alpha, -lam * (-x)**beta)

def prob_weight(p, gamma=0.65):
    """Prelec probability weighting function"""
    return p**gamma / (p**gamma + (1-p)**gamma)**(1/gamma)

def expected_utility(x, alpha=0.88, beta=0.88, lam=2.25):
    """Standard expected utility (CRRA, gamma=0.5)"""
    return np.where(x >= 0, x**0.5, -(-x)**0.5)

x = np.linspace(-3, 3, 500)
probs = np.linspace(0.01, 0.99, 300)

# Different loss aversion parameters
lambdas = [1.0, 1.5, 2.25, 3.5, 5.0]

# 3D: Loss aversion landscape over (x, lambda)
x_3d = np.linspace(-2, 2, 80)
lam_3d = np.linspace(0.5, 6, 80)
X3, L3 = np.meshgrid(x_3d, lam_3d)
V3 = kt_value(X3, lam=L3)

fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35,
                       left=0.06, right=0.97, top=0.90, bottom=0.07)

# 1. Value function surface (3D)
ax1 = fig.add_subplot(gs[0, :2], projection='3d', facecolor='#0a0a0f')
surf = ax1.plot_surface(X3, L3, V3, cmap=PLASMA_CMAP, alpha=0.85, linewidth=0)
ax1.plot_wireframe(X3, L3, V3, color='#ffffff', linewidth=0.08, alpha=0.1)
ax1.contourf(X3, L3, V3, zdir='z', offset=V3.min()-0.5, cmap=PLASMA_CMAP, alpha=0.3, levels=15)
ax1.set_xlabel('Outcome x')
ax1.set_ylabel('Loss Aversion λ')
ax1.set_zlabel('Prospect Value v(x)')
ax1.set_title('PROSPECT VALUE SURFACE v(x, λ)', color=QUANTUM_PINK, pad=12)
ax1.tick_params(colors='#6666aa', labelsize=7)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
ax1.xaxis.pane.set_edgecolor('#1a1a3a')

# 2. Value function slices
ax2 = fig.add_subplot(gs[0, 2])
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(lambdas)))
for lam, col in zip(lambdas, colors):
    ax2.plot(x, kt_value(x, lam=lam), color=col, lw=1.8, label=f'λ={lam}')
ax2.plot(x, expected_utility(x), color=QUANTUM_BLUE, lw=2, ls='--', label='CRRA')
ax2.axhline(0, color='#333355', lw=0.8)
ax2.axvline(0, color='#333355', lw=0.8)
ax2.set_xlabel('Outcome x')
ax2.set_ylabel('Subjective Value')
ax2.set_title('VALUE FUNCTIONS BY λ', color=QUANTUM_PINK)
ax2.legend(fontsize=7)
ax2.set_ylim(-5, 3)

# 3. Probability weighting
ax3 = fig.add_subplot(gs[1, 0])
gammas = [0.4, 0.55, 0.65, 0.80, 1.0]
g_colors = plt.cm.cool(np.linspace(0.1, 0.9, len(gammas)))
for g, col in zip(gammas, g_colors):
    w = prob_weight(probs, gamma=g)
    ax3.plot(probs, w, color=col, lw=1.8, label=f'γ={g}')
ax3.plot([0, 1], [0, 1], color='#555577', ls='--', lw=1, label='Rational')
ax3.fill_between(probs, prob_weight(probs, 0.65), probs,
                 alpha=0.12, color=QUANTUM_PURPLE, label='Overweight tail')
ax3.set_xlabel('True Probability p')
ax3.set_ylabel('Weighted Probability w(p)')
ax3.set_title('PROBABILITY WEIGHTING w(p)', color=QUANTUM_GOLD)
ax3.legend(fontsize=7)

# 4. Gold options mispricing implication
ax4 = fig.add_subplot(gs[1, 1])
K_range = np.linspace(S0 * 0.8, S0 * 1.2, 100)
# OTM put prices under rational vs prospect theory
put_rational = np.maximum(K_range - S0, 0) + 0.3 * np.exp(-0.5*((K_range - S0*0.9)/(S0*0.1))**2)
put_prospect = put_rational * (1 + 0.6 * np.exp(-0.3 * (K_range - S0*0.8)/(S0*0.1)))
ax4.fill_between(K_range, put_rational, put_prospect, alpha=0.3, color=QUANTUM_GOLD)
ax4.plot(K_range, put_rational, color=QUANTUM_BLUE, lw=2, label='Rational Price')
ax4.plot(K_range, put_prospect, color=QUANTUM_RED, lw=2, label='Prospect Theory Price')
ax4.axvline(S0, color=QUANTUM_GOLD, ls='--', lw=1, alpha=0.6, label=f'ATM (${S0:.0f})')
ax4.set_xlabel('Strike K')
ax4.set_ylabel('Put Value')
ax4.set_title('GLD OPTIONS MISPRICING: PT vs RATIONAL', color=QUANTUM_RED)
ax4.legend(fontsize=7)

# 5. CE vs EV
ax5 = fig.add_subplot(gs[1, 2])
gambles_ev = np.linspace(0, 3, 100)
ce_prospect = kt_value(gambles_ev)**(1/0.88)
ce_rational = gambles_ev ** 2
risk_premium = gambles_ev - np.abs(ce_prospect)
ax5.plot(gambles_ev, gambles_ev, color='#555577', ls='--', lw=1, label='EV (Risk Neutral)')
ax5.plot(gambles_ev, np.abs(ce_prospect), color=QUANTUM_PINK, lw=2, label='Prospect CE')
ax5.fill_between(gambles_ev, np.abs(ce_prospect), gambles_ev,
                 alpha=0.2, color=QUANTUM_GOLD, label='Risk Premium')
ax5.set_xlabel('Expected Value')
ax5.set_ylabel('Certainty Equivalent')
ax5.set_title('CERTAINTY EQUIVALENT vs EV (GAINS)', color=QUANTUM_GREEN)
ax5.legend(fontsize=7)

fig.suptitle('MODULE 03 · PROSPECT THEORY · α=β=0.88 λ=2.25 γ=0.65 (Tversky & Kahneman 1992)',
             color='#ffffff', fontsize=9, y=0.97)
plt.show()
print('✓ Prospect theory module rendered')

# %% [markdown]
# ---
# ## MODULE 04 — Market Microstructure & Order Flow
# ### Almgren-Chriss Optimal Execution + VPIN Toxicity (GLD-Calibrated)

# %% Cell 6: Module 04 - Market Microstructure
def simulate_lob_from_real_data(prices, returns, n=None, seed=7):
    """Simulate Limit Order Book state based on real GLD volatility patterns"""
    np.random.seed(seed)
    
    # FIX: Align array lengths to prevent broadcasting errors (returns is 1 shorter than prices)
    min_len = min(len(prices), len(returns))
    if n is None or n > min_len:
        n = min_len
        
    prices_aligned = prices[-n:]
    returns_aligned = returns[-n:]
    
    # Use actual GLD volatility to calibrate spread
    vol_series = pd.Series(returns_aligned).rolling(21).std().bfill().values
    spread = np.clip(vol_series * 2, 0.005, 0.1)
    
    mid = prices_aligned
    
    # Volume correlated with volatility (higher vol = higher volume)
    base_vol = 500
    buy_vol = np.abs(np.random.normal(base_vol, 150, n)) * (1 + vol_series * 10)
    sell_vol = np.abs(np.random.normal(base_vol, 150, n)) * (1 + vol_series * 10)
    
    # Inject information events based on large price moves
    large_moves = np.abs(returns_aligned) > np.percentile(np.abs(returns_aligned), 95)
    for i, is_large in enumerate(large_moves):
        if is_large and i + 20 < n:
            direction = np.sign(returns_aligned[i])
            buy_vol[i:i+20] += direction * np.abs(np.random.normal(800, 200, min(20, n-i)))
            sell_vol[i:i+20] -= direction * np.abs(np.random.normal(400, 100, min(20, n-i)))

    buy_vol = np.abs(buy_vol)
    sell_vol = np.abs(sell_vol)
    ofi = buy_vol - sell_vol
    vpin = np.abs(ofi) / (buy_vol + sell_vol + 1e-8)
    vpin_roll = pd.Series(vpin).rolling(50).mean().values
    
    return mid, spread, buy_vol, sell_vol, ofi, vpin, vpin_roll

def almgren_chriss_trajectory(X0, T, N, sigma, eta, lam):
    """Optimal liquidation trajectory under Almgren-Chriss (2001)"""
    kappa = np.sqrt(lam * sigma**2 / eta)
    t_arr = np.linspace(0, T, N+1)
    x_arr = X0 * np.sinh(kappa*(T - t_arr)) / (np.sinh(kappa*T) + 1e-10)
    trades = -np.diff(x_arr)
    return t_arr, x_arr, trades

# ── Generate data from live GLD ─────────────────────────────────
mid, spread, bv, sv, ofi, vpin, vpin_r = simulate_lob_from_real_data(GLD_PRICES, GLD_RETURNS)
t_arr_ac, x_ac, trades_ac = almgren_chriss_trajectory(
    X0=100000, T=1.0, N=100, sigma=GLD_SIGMA, eta=0.001, lam=10**-6)

fig = plt.figure(figsize=(18, 13), facecolor='#0a0a0f')
gs = gridspec.GridSpec(3, 3, hspace=0.48, wspace=0.35,
                       left=0.06, right=0.97, top=0.92, bottom=0.06)
t = np.arange(len(mid))

# 1. Mid price + spread
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(mid, color=QUANTUM_GOLD, lw=1.2, label='GLD Mid Price')
ax1.fill_between(t, mid - spread*50, mid + spread*50, alpha=0.2, color=QUANTUM_GOLD)
vpin_high = vpin_r > np.nanpercentile(vpin_r, 80)
ax1.fill_between(t, mid.min()-0.2, mid.max()+0.2,
                 where=vpin_high, alpha=0.25, color=QUANTUM_RED, label='High Toxicity (VPIN>P80)')
ax1.set_title('GLD MID PRICE + SPREAD + TOXICITY OVERLAY', color=QUANTUM_GOLD)
ax1.legend(fontsize=8, loc='upper left')

# 2. Order Flow Imbalance
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(t[::4], ofi[::4], width=3,
        color=[QUANTUM_GREEN if v > 0 else QUANTUM_RED for v in ofi[::4]], alpha=0.7)
ax2.plot(pd.Series(ofi).rolling(30).mean(), color=QUANTUM_GOLD, lw=2, label='OFI 30-bar MA')
ax2.axhline(0, color='#444466', lw=0.8)
ax2.set_title('ORDER FLOW IMBALANCE (OFI)', color=QUANTUM_GOLD)
ax2.legend(fontsize=8)

# 3. VPIN
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(vpin_r, color=QUANTUM_RED, lw=1.5, label='VPIN (50-bar roll)')
p80 = np.nanpercentile(vpin_r, 80)
ax3.axhline(p80, color=QUANTUM_GOLD, ls='--', lw=1, label=f'P80 = {p80:.3f}')
ax3.fill_between(t, p80, vpin_r, where=(vpin_r > p80), alpha=0.25, color=QUANTUM_RED)
ax3.set_title('VPIN — TOXICITY INDICATOR', color=QUANTUM_RED)
ax3.legend(fontsize=8)

# 4. Buy vs Sell volume
ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(pd.Series(bv).rolling(20).mean(), color=QUANTUM_GREEN, lw=1.5, label='Buy Volume (MA)')
ax4.plot(pd.Series(sv).rolling(20).mean(), color=QUANTUM_RED, lw=1.5, label='Sell Volume (MA)')
ax4.fill_between(t, pd.Series(bv).rolling(20).mean(), pd.Series(sv).rolling(20).mean(),
                 alpha=0.15, color=QUANTUM_GOLD)
ax4.set_title('BUY vs SELL VOLUME (ROLLING)', color=QUANTUM_GREEN)
ax4.legend(fontsize=8)

# 5. Almgren-Chriss trajectories (3D) calibrated to GLD vol
ax5 = fig.add_subplot(gs[2, :2], projection='3d', facecolor='#0a0a0f')
lam_vals = [1e-7, 5e-7, 1e-6, 5e-6, 1e-5]
lam_cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(lam_vals)))
for lam_v, col in zip(lam_vals, lam_cols):
    t2, x2, _ = almgren_chriss_trajectory(100000, 1.0, 100, GLD_SIGMA, 0.001, lam_v)
    ax5.plot(t2, [lam_v*1e7]*len(t2), x2/1000, color=col, lw=2.0, alpha=0.9)
ax5.set_xlabel('Time')
ax5.set_ylabel('Risk Aversion λ (×1e7)')
ax5.set_zlabel('Inventory (k shares)')
ax5.set_title(f'ALMGREN-CHRISS OPTIMAL TRAJECTORIES (σ_GLD={GLD_SIGMA*100:.2f}%)', color=QUANTUM_PURPLE, pad=12)
ax5.tick_params(colors='#6666aa', labelsize=7)
ax5.xaxis.pane.fill = ax5.yaxis.pane.fill = ax5.zaxis.pane.fill = False

# 6. Trade schedule
ax6 = fig.add_subplot(gs[2, 2])
ax6.bar(t_arr_ac[:-1]*100, trades_ac/1000, color=QUANTUM_GOLD, alpha=0.8, width=0.8)
ax6.plot(t_arr_ac[:-1]*100, np.cumsum(trades_ac)/1000,
         color=QUANTUM_BLUE, lw=2, label='Cumulative')
ax6.set_xlabel('Time (%)')
ax6.set_ylabel('Shares (k)')
ax6.set_title('OPTIMAL LIQUIDATION SCHEDULE', color=QUANTUM_GOLD)
ax6.legend(fontsize=8)

fig.suptitle('MODULE 04 · MARKET MICROSTRUCTURE · Almgren-Chriss + VPIN + Order Flow (GLD)',
             color='#ffffff', fontsize=10, y=0.97)
plt.show()
print('✓ Microstructure module rendered with GLD data')


# %% [markdown]
# ---
# ## MODULE 05 — Kelly Criterion: Optimal Bet Sizing
# ### Fractional Kelly + Edge vs Variance Surface (3D) - GLD Calibrated

# %% Cell 7: Module 05 - Kelly Criterion
def kelly_growth_rate(f, mu, sigma):
    """Expected log-growth rate under continuous Kelly"""
    return f*mu - 0.5*(f*sigma)**2

def max_drawdown_kelly(f, sigma, T=252):
    """Expected maximum drawdown approximation under Kelly-f"""
    return -np.sqrt(2 * sigma**2 * f**2 * T * np.log(2))

# 3D Kelly surface: f vs (mu, sigma)
mu_r = np.linspace(0.0001, 0.003, 60)
sigma_r = np.linspace(0.005, 0.03, 60)
MU, SIG = np.meshgrid(mu_r, sigma_r)
F_OPT = MU / (SIG**2 + 1e-10)  # optimal Kelly fraction
G_OPT = kelly_growth_rate(F_OPT, MU, SIG)  # growth at optimal

# Use actual GLD parameters
mu_gld = GLD_MU
sig_gld = GLD_SIGMA
f_star = mu_gld / (sig_gld**2 + 1e-10)
f_star_used = kelly_cap(f_star)   # 25% Kelly, hard cap 20% of capital

# Fractional Kelly comparison
f_range = np.linspace(0, max(3, abs(f_star) * 3), 500)
growth = kelly_growth_rate(f_range, mu_gld, sig_gld)

# Simulation comparison
n_sim, n_periods = 500, min(252*2, len(GLD_RETURNS))
fracs = [0.25, 0.5, 1.0, 1.5, 2.0]
np.random.seed(99)

# Bootstrap from actual GLD returns
boot_indices = np.random.choice(len(GLD_RETURNS), (n_periods, n_sim))
daily_r = GLD_RETURNS[boot_indices]

fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35,
                       left=0.06, right=0.97, top=0.91, bottom=0.07)

# 1. 3D Kelly surface
ax1 = fig.add_subplot(gs[0, :2], projection='3d', facecolor='#0a0a0f')
surf = ax1.plot_surface(MU*1000, SIG*100, F_OPT.clip(0, 20),
                        cmap=NEON_CMAP, alpha=0.85, linewidth=0)
ax1.contourf(MU*1000, SIG*100, F_OPT.clip(0, 20), zdir='z', offset=-1,
             cmap=NEON_CMAP, alpha=0.3, levels=12)
# Mark GLD position
ax1.scatter([mu_gld*1000], [sig_gld*100], [min(f_star, 20)], 
            color=QUANTUM_GOLD, s=100, marker='*', label='GLD')
ax1.set_xlabel('Daily Edge μ (×1000)')
ax1.set_ylabel('Daily Vol σ (%)')
ax1.set_zlabel('Optimal Kelly f*')
ax1.set_title('OPTIMAL KELLY FRACTION f*(μ,σ) — GLD MARKED', color=QUANTUM_GOLD, pad=12)
ax1.tick_params(colors='#6666aa', labelsize=7)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
fig.colorbar(surf, ax=ax1, shrink=0.35, pad=0.08).set_label('f*', color='#8888cc')

# 2. Growth rate vs f
ax2 = fig.add_subplot(gs[0, 2])
if f_star != 0:
    ax2.plot(f_range / abs(f_star), growth * 252, color=QUANTUM_GREEN, lw=2.5)
    ax2.axvline(1.0, color=QUANTUM_GOLD, ls='--', lw=1.5, label='Full Kelly')
    ax2.axvline(0.5, color=QUANTUM_BLUE, ls=':', lw=1.5, label='Half Kelly')
    ax2.fill_between(f_range/abs(f_star), 0, growth*252,
                     where=(f_range/abs(f_star) <= 1), alpha=0.15, color=QUANTUM_GREEN)
    ax2.fill_between(f_range/abs(f_star), growth*252, 0,
                     where=(f_range/abs(f_star) > 1.8), alpha=0.2, color=QUANTUM_RED)
    ax2.set_xlabel('Fraction of Kelly (f / f*)')
else:
    ax2.plot(f_range, growth * 252, color=QUANTUM_GREEN, lw=2.5)
    ax2.set_xlabel('Kelly Fraction f')
ax2.axhline(0, color='#444466', lw=0.8)
ax2.set_ylabel('Ann. Growth Rate')
ax2.set_title(f'GLD GROWTH RATE vs KELLY FRACTION (f*={f_star:.2f})', color=QUANTUM_GREEN)
ax2.legend(fontsize=8)
ax2.set_xlim(0, 3)

# 3. Equity curves by fraction
ax3 = fig.add_subplot(gs[1, :2])
frac_cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(fracs)))
for frac, col in zip(fracs, frac_cols):
    f_use = frac * f_star
    log_ret = np.log1p(np.clip(f_use * daily_r, -0.99, 10))
    cum_med = np.exp(np.median(np.cumsum(log_ret, axis=0), axis=1))
    cum_p25 = np.exp(np.percentile(np.cumsum(log_ret, axis=0), 25, axis=1))
    cum_p75 = np.exp(np.percentile(np.cumsum(log_ret, axis=0), 75, axis=1))
    t_plot = np.arange(n_periods)
    ax3.plot(t_plot, cum_med, color=col, lw=1.8, label=f'{frac}× Kelly')
    ax3.fill_between(t_plot, cum_p25, cum_p75, color=col, alpha=0.06)
ax3.axhline(1, color='#444466', lw=0.8, ls='--')
ax3.set_yscale('log')
ax3.set_xlabel('Trading Days')
ax3.set_ylabel('Wealth (log scale)')
ax3.legend(fontsize=8, ncol=3)
ax3.set_title('GLD WEALTH PATHS BY KELLY FRACTION — 500 SIMULATIONS (MEDIAN ± IQR)', color=QUANTUM_GOLD)

# 4. Ruin probability vs fraction
ax4 = fig.add_subplot(gs[1, 2])
frac_scan = np.linspace(0.1, 3.0, 100)
ruin_prob = []
for frac in frac_scan:
    f_use = frac * f_star
    log_ret = np.log1p(np.clip(f_use * daily_r, -0.99, 10))
    final = np.exp(np.sum(log_ret, axis=0))
    ruin_prob.append(np.mean(final < 0.5))
ax4.plot(frac_scan, ruin_prob, color=QUANTUM_RED, lw=2.5)
ax4.fill_between(frac_scan, 0, ruin_prob, alpha=0.15, color=QUANTUM_RED)
ax4.axvline(1, color=QUANTUM_GOLD, ls='--', lw=1.5, label='Full Kelly')
ax4.axvline(0.5, color=QUANTUM_BLUE, ls=':', lw=1.5, label='Half Kelly')
ax4.set_xlabel('Kelly Fraction (× f*)')
ax4.set_ylabel('P(Wealth < 50%)')
ax4.set_title('GLD RUIN PROBABILITY vs KELLY FRACTION', color=QUANTUM_RED)
ax4.legend(fontsize=8)

fig.suptitle(f'MODULE 05 · KELLY CRITERION · GLD: μ={mu_gld*252:.2%} p.a. σ={sig_gld*np.sqrt(252):.2%} p.a. f*={f_star:.2f}',
             color='#ffffff', fontsize=9, y=0.97)
plt.show()
print(f'✓ Kelly module | GLD raw f* = {f_star:.2f} -> USE {f_star_used:.1%} of capital')

# %% [markdown]
# ---
# ## MODULE 06 — PCA Factor Decomposition
# ### Statistical Risk Factors from GLD Return Space

# %% Cell 8: Module 06 - PCA Factor Decomposition
np.random.seed(21)

# Create a synthetic universe of gold-related assets based on GLD
n_assets = 30
n_periods = len(GLD_RETURNS)

# Generate correlated assets based on GLD (gold miners, other commodities, etc.)
# Factor 1: Market/Gold factor (high correlation with GLD)
# Factor 2: Style factor (momentum vs value)
# Factor 3: Idiosyncratic

# Create loadings
B = np.random.normal(0, 1, (n_assets, 3))
B[:, 0] = np.abs(B[:, 0]) * 0.8 + 0.2  # Market factor - all positive, correlated with gold

# Generate factor returns using GLD as the primary driver
F = np.zeros((n_periods, 3))
F[:, 0] = GLD_RETURNS * 1.5 + np.random.normal(0, GLD_SIGMA * 0.3, n_periods)  # Gold/Market factor
F[:, 1] = np.random.normal(0, GLD_SIGMA * 0.6, n_periods)  # Style factor
F[:, 2] = np.random.normal(0, GLD_SIGMA * 0.4, n_periods)  # Residual factor

eps = np.random.normal(0, GLD_SIGMA * 0.3, (n_periods, n_assets))
R = F @ B.T + eps

# PCA
cov_mat = np.cov(R.T)
evals, evecs = eigh(cov_mat)
evals = evals[::-1]
evecs = evecs[:, ::-1]  # Sort descending
var_explained = evals / evals.sum()
cum_var = np.cumsum(var_explained)
factors_pca = R @ evecs[:, :3]

fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(2, 3, hspace=0.44, wspace=0.35,
                       left=0.06, right=0.97, top=0.91, bottom=0.07)

# 1. 3D Factor space
ax1 = fig.add_subplot(gs[0, :2], projection='3d', facecolor='#0a0a0f')
scatter_col = plt.cm.plasma(np.linspace(0, 1, n_periods))
ax1.scatter(factors_pca[:, 0], factors_pca[:, 1], factors_pca[:, 2],
            c=scatter_col, s=2, alpha=0.6)
# Factor loading arrows
scale = 0.3
for i in range(min(15, n_assets)):
    ax1.quiver(0, 0, 0, evecs[i,0]*scale, evecs[i,1]*scale, evecs[i,2]*scale,
               color=QUANTUM_GOLD, alpha=0.5, arrow_length_ratio=0.2, linewidth=1)
ax1.set_xlabel('PC1 (Gold/Market)')
ax1.set_ylabel('PC2 (Style)')
ax1.set_zlabel('PC3')
ax1.set_title('3D FACTOR SPACE — GOLD UNIVERSE PC1/PC2/PC3', color=QUANTUM_GOLD, pad=12)
ax1.tick_params(colors='#6666aa', labelsize=7)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False

# 2. Scree plot
ax2 = fig.add_subplot(gs[0, 2])
top_k = 15
bars = ax2.bar(range(1, top_k+1), var_explained[:top_k]*100,
               color=[QUANTUM_GOLD if i < 3 else '#3a3a6a' for i in range(top_k)],
               alpha=0.85, edgecolor='none')
ax2b = ax2.twinx()
ax2b.plot(range(1, top_k+1), cum_var[:top_k]*100, color=QUANTUM_BLUE, lw=2, marker='o', markersize=4)
ax2b.axhline(80, color=QUANTUM_GREEN, ls='--', lw=1, alpha=0.7)
ax2b.set_ylabel('Cumulative Variance (%)', color=QUANTUM_BLUE)
ax2b.tick_params(colors='#8888cc')
ax2.set_xlabel('Principal Component')
ax2.set_ylabel('% Variance Explained')
ax2.set_title('SCREE PLOT — EIGENVALUE SPECTRUM', color=QUANTUM_PURPLE)
for i, bar in enumerate(bars[:3]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{var_explained[i]*100:.1f}%', ha='center', color=QUANTUM_GOLD, fontsize=8)

# 3. Factor loading heatmap (top factors)
ax3 = fig.add_subplot(gs[1, 0])
loading_mat = evecs[:15, :5].T
im = ax3.imshow(loading_mat, cmap='RdYlGn', aspect='auto',
                vmin=-loading_mat.max(), vmax=loading_mat.max())
ax3.set_xlabel('Asset Index')
ax3.set_ylabel('Factor')
ax3.set_yticks(range(5))
ax3.set_yticklabels(['PC1', 'PC2', 'PC3', 'PC4', 'PC5'])
ax3.set_title('FACTOR LOADING MATRIX (Top 5 PCs)', color=QUANTUM_GOLD)
plt.colorbar(im, ax=ax3, shrink=0.7).set_label('Loading', color='#8888cc', fontsize=8)

# 4. Factor return time series
ax4 = fig.add_subplot(gs[1, 1])
f_cols = [QUANTUM_GOLD, QUANTUM_GREEN, QUANTUM_PURPLE]
f_names = ['PC1 (Gold/Market)', 'PC2 (Style)', 'PC3 (Residual)']
for i, (col, name) in enumerate(zip(f_cols, f_names)):
    cum = np.cumsum(factors_pca[:, i])
    cum = cum / (cum.std() + 1e-8) * 3  # normalize for display
    ax4.plot(cum, color=col, lw=1.2, label=name, alpha=0.85)
ax4.axhline(0, color='#333355', lw=0.7)
ax4.set_title('CUMULATIVE FACTOR RETURNS', color=QUANTUM_GREEN)
ax4.legend(fontsize=8)

# 5. Correlation with GLD
ax5 = fig.add_subplot(gs[1, 2])
# Calculate correlation of each synthetic asset with GLD
gld_corr = [np.corrcoef(R[:, i], GLD_RETURNS)[0, 1] for i in range(n_assets)]
sort_idx = np.argsort(gld_corr)[::-1]
colors_corr = [QUANTUM_GOLD if c > 0.5 else QUANTUM_BLUE if c > 0 else QUANTUM_RED for c in np.array(gld_corr)[sort_idx]]
ax5.barh(range(n_assets), np.array(gld_corr)[sort_idx], color=colors_corr, alpha=0.8)
ax5.axvline(0, color='#555577', lw=0.8)
ax5.axvline(0.5, color=QUANTUM_GOLD, ls='--', lw=1, alpha=0.7)
ax5.set_xlabel('Correlation with GLD')
ax5.set_ylabel('Asset (sorted)')
ax5.set_title('ASSET CORRELATION WITH GLD', color=QUANTUM_GOLD)

fig.suptitle(f'MODULE 06 · PCA FACTOR DECOMPOSITION · PC1={var_explained[0]*100:.1f}% PC2={var_explained[1]*100:.1f}% PC3={var_explained[2]*100:.1f}%',
             color='#ffffff', fontsize=9, y=0.97)
plt.show()
print(f'✓ PCA: Top 3 PCs explain {cum_var[2]*100:.1f}% of variance')

# %% [markdown]
# ---
# ## MODULE 07 — Merton Jump-Diffusion Model
# ### Quantum Probability Density Surface (Calibrated to GLD Tail Risk)

# %% Cell 9: Module 07 - Merton Jump Diffusion
def merton_density(x, mu, sigma, lam, mu_j, sigma_j, T=1/252, n_terms=20):
    """PDF of Merton jump-diffusion log-return"""
    pdf = np.zeros_like(x, dtype=float)
    k = np.exp(mu_j + 0.5*sigma_j**2) - 1
    drift = (mu - 0.5*sigma**2 - lam*k) * T
    for n in range(n_terms):
        w = np.exp(-lam*T) * (lam*T)**n / math.factorial(n)
        mu_n = drift + n * mu_j
        sig_n = np.sqrt(sigma**2 * T + n * sigma_j**2)
        if sig_n > 0:
            pdf += w * norm.pdf(x, mu_n, sig_n)
    return pdf

# Estimate jump parameters from GLD data
# Look for large moves (potential jumps)
threshold = np.percentile(np.abs(GLD_LOG_RETURNS), 95)
jumps = GLD_LOG_RETURNS[np.abs(GLD_LOG_RETURNS) > threshold]
mu_j = np.mean(jumps) if len(jumps) > 0 else -0.02
sigma_j = np.std(jumps) if len(jumps) > 1 else 0.01

# Estimate jump intensity (jumps per year)
jump_count = len(jumps)
years = len(GLD_LOG_RETURNS) / 252
lam_estimated = jump_count / years if years > 0 else 10

print(f'GLD Jump Analysis:')
print(f'  Estimated jump intensity λ: {lam_estimated:.1f} jumps/year')
print(f'  Estimated jump mean μ_J: {mu_j*100:.2f}%')
print(f'  Estimated jump vol σ_J: {sigma_j*100:.2f}%')

# Parameters
x_vals = np.linspace(-0.15, 0.10, 200)
mu_base = GLD_MU
sig_base = GLD_SIGMA

# 3D: density surface over (x, lambda)
lam_vals_3d = np.linspace(0.5, 30, 50)
X3d, L3d = np.meshgrid(x_vals, lam_vals_3d)
Z3d = np.zeros_like(X3d)
for i, lam_v in enumerate(lam_vals_3d):
    Z3d[i] = merton_density(x_vals, mu_base, sig_base, lam_v, mu_j, sigma_j)

fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(2, 3, hspace=0.44, wspace=0.35,
                       left=0.06, right=0.97, top=0.91, bottom=0.07)

# 1. 3D Density surface
ax1 = fig.add_subplot(gs[0, :2], projection='3d', facecolor='#0a0a0f')
surf = ax1.plot_surface(X3d*100, L3d, Z3d, cmap=FIRE_CMAP, alpha=0.85, linewidth=0)
ax1.contourf(X3d*100, L3d, Z3d, zdir='z', offset=-0.5, cmap=FIRE_CMAP, alpha=0.3, levels=15)
ax1.set_xlabel('Log-Return (%)')
ax1.set_ylabel('Jump Intensity λ')
ax1.set_zlabel('Density')
ax1.set_title('GLD MERTON JUMP-DIFFUSION DENSITY SURFACE', color=QUANTUM_GOLD, pad=12)
ax1.tick_params(colors='#6666aa', labelsize=7)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
fig.colorbar(surf, ax=ax1, shrink=0.35, pad=0.08).set_label('f(x)', color='#8888cc')

# 2. Density slices (cross-sections)
ax2 = fig.add_subplot(gs[0, 2])
lam_slices = [0, 5, 10, 15, 25]
s_cols = plt.cm.hot(np.linspace(0.2, 0.9, len(lam_slices)))
gbm_pdf = norm.pdf(x_vals, mu_base - 0.5*sig_base**2, sig_base)
ax2.plot(x_vals*100, gbm_pdf, color=QUANTUM_BLUE, lw=2.5, ls='--', label='Pure GBM (λ=0)')
for lam_v, col in zip(lam_slices[1:], s_cols[1:]):
    pdf = merton_density(x_vals, mu_base, sig_base, lam_v, mu_j, sigma_j)
    ax2.plot(x_vals*100, pdf, color=col, lw=1.8, label=f'λ={lam_v}')
ax2.set_xlabel('Log-Return (%)')
ax2.set_ylabel('Density')
ax2.set_title('GLD DENSITY BY JUMP INTENSITY λ', color=QUANTUM_GOLD)
ax2.legend(fontsize=8)
ax2.set_yscale('log')
ax2.set_ylim(1e-30, 1e2)

# 3. Actual GLD returns histogram vs fitted distributions
ax3 = fig.add_subplot(gs[1, 0])
bins = np.linspace(-0.08, 0.08, 80)
ax3.hist(GLD_LOG_RETURNS, bins=bins, density=True, alpha=0.6, color=QUANTUM_GOLD, label='Actual GLD')
ax3.plot(x_vals*100/100, norm.pdf(x_vals, mu_base, sig_base), color=QUANTUM_BLUE, lw=2, ls='--', label='Normal')
ax3.plot(x_vals*100/100, merton_density(x_vals, mu_base, sig_base, lam_estimated, mu_j, sigma_j),
         color=QUANTUM_RED, lw=2, label=f'Merton (λ={lam_estimated:.0f})')
ax3.set_xlabel('Log-Return')
ax3.set_ylabel('Density')
ax3.set_title('GLD EMPIRICAL vs FITTED DISTRIBUTIONS', color=QUANTUM_GOLD)
ax3.legend(fontsize=8)
ax3.set_yscale('log')
ax3.set_ylim(1e-3, 1e3)

# 4. Simulated paths with jumps
ax4 = fig.add_subplot(gs[1, 1])
np.random.seed(33)
n_paths, n_days = 5, 252
path_cols = plt.cm.plasma(np.linspace(0.1, 0.9, n_paths))
for pc in path_cols:
    dt = 1/252
    rets = np.random.normal((mu_base - 0.5*sig_base**2)*dt, sig_base*np.sqrt(dt), n_days)
    # Add jumps
    n_jumps = np.random.poisson(lam_estimated * dt, n_days)
    for t_j, nj in enumerate(n_jumps):
        if nj > 0:
            rets[t_j] += np.random.normal(mu_j, sigma_j, nj).sum()
    ax4.plot(S0 * np.exp(np.cumsum(rets)), color=pc, lw=1.2, alpha=0.8)
ax4.axhline(S0, color='#444466', lw=0.8, ls='--', label=f'S₀=${S0:.0f}')
ax4.set_xlabel('Trading Day')
ax4.set_ylabel('Price ($)')
ax4.set_title(f'GLD MERTON SAMPLE PATHS (λ={lam_estimated:.0f}, μ_J={mu_j:.3f})', color=QUANTUM_GOLD)
ax4.legend(fontsize=8)

# 5. Tail risk comparison
ax5 = fig.add_subplot(gs[1, 2])
n_mc = 100000
gbm_r = np.random.normal(mu_base - 0.5*sig_base**2, sig_base, n_mc)
# Jump-diffusion
jd_r = np.random.normal(mu_base - 0.5*sig_base**2, sig_base, n_mc)
nj_mc = np.random.poisson(lam_estimated/252, n_mc)
for i in range(n_mc):
    if nj_mc[i] > 0:
        jd_r[i] += np.random.normal(mu_j, sigma_j, nj_mc[i]).sum()

q_range = np.linspace(0.001, 0.05, 100)
var_gbm = np.quantile(gbm_r, q_range)
var_jd = np.quantile(jd_r, q_range)
ax5.plot(q_range*100, -var_gbm*100, color=QUANTUM_BLUE, lw=2, label='GBM VaR')
ax5.plot(q_range*100, -var_jd*100, color=QUANTUM_RED, lw=2, label='Jump-Diffusion VaR')
ax5.fill_between(q_range*100, -var_gbm*100, -var_jd*100,
                 alpha=0.2, color=QUANTUM_GOLD, label='Tail Risk Premium')
ax5.set_xlabel('Confidence Level (%)')
ax5.set_ylabel('VaR (%) [Loss]')
ax5.set_title('GLD TAIL RISK: GBM vs JUMP-DIFFUSION VaR', color=QUANTUM_RED)
ax5.legend(fontsize=8)

fig.suptitle(f'MODULE 07 · MERTON JUMP-DIFFUSION · GLD: λ={lam_estimated:.0f} jumps/yr μ_J={mu_j*100:.1f}% σ_J={sigma_j*100:.1f}%',
             color='#ffffff', fontsize=9, y=0.97)
plt.show()
print('✓ Merton jump-diffusion module rendered with GLD calibration')

# %% [markdown]
# ---
# ## MODULE 08 — Information Cascades & Herding
# ### Bayesian Updating + Social Learning Breakdown (Gold Market Context)
# %% Cell 10: Module 08 - Information Cascades
def simulate_cascade(n_agents=200, p_signal=0.65, seed=55):
    """Bikhchandani-Hirshleifer-Welch (1992) information cascade"""
    np.random.seed(seed)
    true_state = 1  # True state is HIGH (gold bullish)
    beliefs = [0.5]
    actions = []
    cascade_start = None

    for i in range(n_agents):
        prior = beliefs[-1]
        # Private signal
        correct = np.random.rand() < p_signal
        signal = true_state if correct else (1 - true_state)

        # Bayesian update
        if signal == 1:
            posterior = (prior * p_signal) / (prior * p_signal + (1-prior) * (1-p_signal) + 1e-8)
        else:
            posterior = (prior * (1-p_signal)) / (prior * (1-p_signal) + (1-prior) * p_signal + 1e-8)

        # Action: follow crowd if cascade; follow signal if not
        n_buy = sum(1 for a in actions if a == 1)
        n_sell = sum(1 for a in actions if a == 0)

        if n_buy > n_sell + 2:  # Strong cascade: buy regardless
            action = 1
            if cascade_start is None:
                cascade_start = i
        elif n_sell > n_buy + 2:  # Strong cascade: sell regardless
            action = 0
            if cascade_start is None:
                cascade_start = i
        else:
            action = 1 if posterior > 0.5 else 0
            cascade_start = None  # Reset if private info used

        beliefs.append(posterior)
        actions.append(action)

    return np.array(beliefs), np.array(actions)

def simulate_herding_price(base_prices, p_herd=0.0, seed=42):
    """Price process with herding component based on real GLD"""
    np.random.seed(seed)
    n = len(base_prices)
    prices = [base_prices[0]]
    herd_mom = 0.0
    
    for t in range(1, n):
        # Base return from actual GLD
        fundamental_return = (base_prices[t] / base_prices[t-1]) - 1
        
        if np.random.rand() < p_herd:
            mean_price = np.mean(prices[max(0, len(prices)-20):]) if len(prices) > 1 else prices[-1]
            herd_mom = 0.8 * herd_mom + 0.2 * np.sign(prices[-1] - mean_price)
            price_move = fundamental_return + herd_mom * 0.003
        else:
            herd_mom *= 0.9
            price_move = fundamental_return
        prices.append(prices[-1] * (1 + price_move))
    
    return np.array(prices)

beliefs, actions = simulate_cascade(n_agents=300, p_signal=0.63)
buy_fraction = pd.Series(actions).rolling(20).mean().values

# Use actual GLD prices as base
p_no_herd = GLD_PRICES.copy()
p_herd_low = simulate_herding_price(GLD_PRICES, p_herd=0.3)
p_herd_hi = simulate_herding_price(GLD_PRICES, p_herd=0.7)

# Autocorrelation comparison
def acf(x, lags=30):
    r = np.array(x)
    r = (r - r.mean()) / (r.std() + 1e-8)
    return [np.corrcoef(r[:-k], r[k:])[0, 1] if len(r) > k else 0 for k in range(1, lags+1)]

ret_no = np.diff(np.log(p_no_herd))
ret_herd = np.diff(np.log(p_herd_hi))
lags = np.arange(1, 31)

fig = plt.figure(figsize=(18, 12), facecolor='#0a0a0f')
gs = gridspec.GridSpec(2, 3, hspace=0.44, wspace=0.35,
                       left=0.06, right=0.97, top=0.91, bottom=0.07)

# 1. Bayesian updating trajectory
ax1 = fig.add_subplot(gs[0, :])
agents = np.arange(len(beliefs))
ax1.plot(beliefs, color=QUANTUM_GOLD, lw=1.8, label='Posterior P(Gold Bullish)', alpha=0.9)
ax1.scatter(agents[1:][actions==1], beliefs[1:][actions==1],
            c=QUANTUM_GREEN, s=8, alpha=0.5, label='Buy Gold')
ax1.scatter(agents[1:][actions==0], beliefs[1:][actions==0],
            c=QUANTUM_RED, s=8, alpha=0.5, label='Sell Gold')

# FIX: Perfectly aligned arrays for fill_between
valid_bf = ~np.isnan(buy_fraction)
if np.any(valid_bf):
    ax1.fill_between(agents[1:][valid_bf], buy_fraction[valid_bf], 0.5,
                     where=(buy_fraction[valid_bf] > 0.5), alpha=0.1, color=QUANTUM_GREEN)
    ax1.fill_between(agents[1:][valid_bf], buy_fraction[valid_bf], 0.5,
                     where=(buy_fraction[valid_bf] < 0.5), alpha=0.1, color=QUANTUM_RED)
    
ax1.axhline(0.5, color='#555577', ls='--', lw=0.8)
ax1.axhline(1.0, color=QUANTUM_GREEN, ls=':', lw=0.7, alpha=0.5, label='True State: Bullish')
ax1.set_xlabel('Agent Index (Sequence)')
ax1.set_ylabel('Posterior Belief')
ax1.set_title('GOLD MARKET BAYESIAN INFORMATION CASCADE — P(Bullish|Signals)', color=QUANTUM_GOLD)
ax1.legend(fontsize=8, ncol=4)
ax1.set_ylim(-0.05, 1.05)

# 2. Herding price comparison
ax2 = fig.add_subplot(gs[1, :2])
ax2.plot(p_no_herd, color=QUANTUM_GOLD, lw=1.5, label='Actual GLD (No Herding)')
ax2.plot(p_herd_low, color=QUANTUM_BLUE, lw=1.5, label='Low Herding (p=0.3)', alpha=0.8)
ax2.plot(p_herd_hi, color=QUANTUM_RED, lw=1.5, label='High Herding (p=0.7)', alpha=0.8)
ax2.set_xlabel('Time')
ax2.set_ylabel('Price ($)')
ax2.set_title('GLD PRICE DYNAMICS WITH VARYING HERDING INTENSITY', color=QUANTUM_GOLD)
ax2.legend(fontsize=8)

# Add vol comparison annotation
vols = [ret_no.std()*np.sqrt(252)*100,
        np.diff(np.log(p_herd_low)).std()*np.sqrt(252)*100,
        ret_herd.std()*np.sqrt(252)*100]
y_pos = [p_no_herd.max() * 1.02, p_no_herd.max() * 0.98, p_no_herd.max() * 0.94]
for v, color, y in zip(vols, [QUANTUM_GOLD, QUANTUM_BLUE, QUANTUM_RED], y_pos):
    ax2.text(5, y, f'Ann.Vol={v:.1f}%', color=color, fontsize=8)

# 3. ACF of returns
ax3 = fig.add_subplot(gs[1, 2])
acf_no = acf(ret_no)
acf_herd = acf(ret_herd)
ax3.bar(lags - 0.2, acf_no, width=0.4, color=QUANTUM_GOLD, alpha=0.8, label='Actual GLD')
ax3.bar(lags + 0.2, acf_herd, width=0.4, color=QUANTUM_RED, alpha=0.8, label='High Herding')
conf = 1.96 / (np.sqrt(len(ret_no)) + 1e-8)
ax3.axhline(conf, color=QUANTUM_BLUE, ls='--', lw=1, alpha=0.7)
ax3.axhline(-conf, color=QUANTUM_BLUE, ls='--', lw=1, alpha=0.7)
ax3.axhline(0, color='#444466', lw=0.8)
ax3.set_xlabel('Lag (days)')
ax3.set_ylabel('Autocorrelation')
ax3.set_title('GLD RETURN ACF: ACTUAL vs HERDING', color=QUANTUM_GOLD)
ax3.legend(fontsize=8)

fig.suptitle('MODULE 08 · INFORMATION CASCADES · Bikhchandani-Hirshleifer-Welch (1992) — Gold Market',
             color='#ffffff', fontsize=9, y=0.97)
plt.show()
print('✓ Herding cascade module rendered with GLD data')


# %% Cell 11: Final Summary Dashboard
fig, axes = plt.subplots(2, 4, figsize=(20, 8), facecolor='#0a0a0f')
fig.patch.set_facecolor('#0a0a0f')

modules = [
    ('HESTON SV', QUANTUM_GOLD, 'GLD Stochastic\nVolatility', 3, 5),
    ('MOMENTUM', QUANTUM_GREEN, 'GLD Regime-Aware\nSignal', 2, 4),
    ('PROSPECT', QUANTUM_PINK, 'Behavioral\nBias Model', 2, 4),
    ('MICRO', QUANTUM_BLUE, 'GLD Order Flow\nToxicity', 3, 5),
    ('KELLY', QUANTUM_PURPLE, 'GLD Optimal\nSizing', 1, 3),
    ('PCA', '#00ccff', 'Gold Factor\nRisk Model', 2, 4),
    ('MERTON', QUANTUM_RED, 'GLD Jump\nDiffusion', 3, 5),
    ('HERDING', '#ffaa00', 'Gold Market\nCascade', 2, 4),
]

for ax, (name, color, desc, stars, total) in zip(axes.flat, modules):
    ax.set_facecolor('#0d0d1a')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(1.5)
    ax.text(5, 7.8, name, ha='center', va='center', color=color,
            fontsize=13, fontweight='bold', fontfamily='monospace')
    ax.text(5, 5.8, desc, ha='center', va='center', color='#aaaacc',
            fontsize=9, fontfamily='monospace')
    stars_str = '★' * stars + '☆' * (total - stars)
    ax.text(5, 3.5, stars_str, ha='center', va='center', color=color, fontsize=14)
    ax.text(5, 1.5, '✓ RENDERED', ha='center', va='center',
            color='#44ff88', fontsize=8, fontfamily='monospace')

fig.suptitle(f'QUANT BEHAVIORAL FINANCE FRAMEWORK SUITE — ALL MODULES COMPLETE | GLD=${GLD_PRICES[-1]:.2f}',
             color='#ffffff', fontsize=13, fontweight='bold', y=1.01,
             fontfamily='monospace')
plt.tight_layout(pad=1.2)
plt.show()

print()
print('╔══════════════════════════════════════════════════════════════╗')
print('║        ALL 8 MODULES SUCCESSFULLY RENDERED WITH GLD          ║')
print('║       Heston · Momentum · Prospect · Microstructure          ║')
print('║       Kelly  · PCA      · Merton   · Information Cascade     ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  GLD Price: ${GLD_PRICES[-1]:.2f}                                       ║')
print(f'║  GLD Ann. Return: {GLD_MU * 252 * 100:.2f}%                                      ║')
print(f'║  GLD Ann. Volatility: {GLD_SIGMA * np.sqrt(252) * 100:.2f}%                               ║')
print('╚══════════════════════════════════════════════════════════════╝')
